## Import necessary modules

In [1]:
from monitor_models import MonitorNet
import torch
import numpy as np
import pickle
from tqdm import tqdm
import random
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots, show
import os

## Set paths - This is the only cell that needs to be changed per user according to paths on their device

In [2]:
# set according to your paths

pred_pkl_path = '/home/ns38942/VERITAS/Monitor/results/preds_Delay_Spread.pkl'

## Read the pickle file containing test results

In [3]:
with open (pred_pkl_path, 'rb') as handle:
    content = pickle.load(handle)

pred_dict = content['pred_dict']
pred_dict_id = content['pred_dict_id']

FileNotFoundError: [Errno 2] No such file or directory: '/home/ns38942/DeepRx/deep-rx-sim-env/Monitor/results/preds_Delay_Spread.pkl'

## Plot test set clusters

In [ ]:
## read the prediction dictionary and study the ultimate layer features
## perform tsne

feature_dim = 256

delay_range = [10.0, 50.0, 80.0, 100.0, 200.0, 400.0]
SNR_range = [0,10,20]

channel_profiles = ['tdl_b']

color_list = ['royalblue','red','ForestGreen','orange','black','violet', 'cyan','gray','yellow','salmon','pink']
plt.rcParams['axes.titley'] = -0.2    # y is in axes-relative coordinates.
fig, axes = plt.subplots(figsize=(15,4), nrows=1, ncols=3)
matplotlib.rcParams.update({'font.size': 12})


for i in tqdm(range(3)): # because we have 3 SNRs and want 3 subplots
    SNR = SNR_range[i]
    ax = axes[i]
    all_classes_array = np.empty(shape=[0, feature_dim])
    stride_list = []
    
    for delay in delay_range:
        this_class_array = np.empty(shape=[0, feature_dim])
        for channel in channel_profiles:
            vectored = np.squeeze(np.array(pred_dict[channel][delay][SNR]))
            this_class_array = np.concatenate((this_class_array, vectored), axis=0)
        stride_list.append(this_class_array.shape[0])
        all_classes_array = np.concatenate((all_classes_array, this_class_array), axis=0)
    
    X = all_classes_array
    print(X.shape)
        
    
    """for tsne """    
    t_sne = TSNE(n_components=2, init='random')
    X_embedded= t_sne.fit_transform(X)
    
    x1 = X_embedded[:stride_list[0],0]
    y1 = X_embedded[:stride_list[0],1]
    start = stride_list[0]
    x2 = X_embedded[start:start+stride_list[1],0]
    y2 = X_embedded[start:start+stride_list[1],1]
    start+= stride_list[1]
    x3 = X_embedded[start:start+stride_list[2],0]
    y3 = X_embedded[start:start+stride_list[2],1]
    start+= stride_list[2]
    x4 = X_embedded[start:start+stride_list[3],0]
    y4 = X_embedded[start:start+stride_list[3],1]
    start+= stride_list[3]
    x5 = X_embedded[start:start+stride_list[4],0]
    y5 = X_embedded[start:start+stride_list[4],1]
    
    
    ax.scatter(x1, y1,s=2, linewidths=1, color=color_list[0], marker='o', label='10 ns (ID)') 
    ax.scatter(x2, y2,s=2, linewidths=1, color=color_list[1], marker='o', label='50 ns (ID)')
    ax.scatter(x3, y3,s=2, linewidths=1, color=color_list[2], marker='o', label='80 ns (ID)')
    ax.scatter(x4, y4,s=2, linewidths=1, color=color_list[3], marker='o', label='200 ns (OOD)')
    ax.scatter(x5, y5,s=2, linewidths=1, color=color_list[4], marker='o', label='400 ns (OOD)')
    
    ax.grid()
    
    if i == len(SNR_range)-1:  # this is the last subplot, we need to insert legend
        legend = plt.legend(bbox_to_anchor=(1.1, 1), loc='upper left', borderaxespad=0, markerscale=9, edgecolor='black')
        legend.get_frame().set_alpha(None)
        legend.get_frame().set_facecolor((0, 0, 0, 0.1))


axes[0].title.set_text('(a) Eb/N0 = 0 dB') 
axes[1].title.set_text('(b) Eb/N0 = 10 dB') 
axes[2].title.set_text('(c) Eb/N0 = 20 dB')

plt.savefig('foo.pdf', bbox_inches='tight')

## Run the OOD detection algorithm and calculate OOD detection rate

In [ ]:
lambda_val = 0.95

k_neighbors = [5,10,15]
feature_dim = 256
print(feature_dim)

def distance_function(a,b):
    a = np.squeeze(a)
    b = np.squeeze(b)
    distance = np.linalg.norm(a-b)    
    return distance


ood_dict = {}

SNR_range = list(np.arange(0,21,step=2))
# SNR_range = [20]
print(SNR_range)
                 
channel = 'tdl_b'
    
for k_neighbor in k_neighbors:
    
    All_SNR_accuracies = []

    for i in tqdm(range(len(SNR_range))): 
        speed_range = [10.0, 50.0, 80.0]  # training delay range
        
        SNR = SNR_range[i]
        all_classes_array = np.empty(shape=[0, feature_dim])
        all_classes_y = np.empty(shape=[0])
        train_set_stride_list = []
        
        for class_index,speed in enumerate(speed_range):
            this_class_array = np.squeeze(np.array(pred_dict_id[channel][speed][SNR]))
            this_class_y = np.ones((this_class_array.shape[0]))*(class_index)
            train_set_stride_list.append(this_class_array.shape[0])
            all_classes_y = np.concatenate((all_classes_y, this_class_y), axis=0)
            all_classes_array = np.concatenate((all_classes_array, this_class_array), axis=0)
        
        X_train = all_classes_array
        y_train = all_classes_y
    
    
        knn = NearestNeighbors(n_neighbors=k_neighbor, metric='euclidean', algorithm='ball_tree')
        knn.fit(X_train, y_train)
    
        
        start = 0 
        center_list, max_distance_list = [], []
        for i in range(len(speed_range)):
            # for each in distribution class
            this_X_train = X_train[start:start+train_set_stride_list[i]]
            start += train_set_stride_list[i]
            this_center = np.mean(this_X_train, axis=0)
            center_list.append(this_center)
            distance_list = []
            for X in this_X_train:
                distance_list.append(distance_function(X,this_center))
                
            # now find the max distance as the distance where 95% of samples are calculated as ID
            distance_list.sort()
            distance_list = distance_list[:int(lambda_val*len(distance_list))]
            max_distance_list.append(np.max(distance_list))
            
        # Now the center of all clusters are ready in center_list
        # and max distance for each cluster is also determined
    
        # go through the test set and for each sample find the nearest neighbors
        all_classes_array = np.empty(shape=[0, feature_dim])
        stride_list = []
    
        speed_range = [10.0, 50.0, 80.0, 200.0, 400.0] # test delay range
        this_SNR_accuracies = []
        for class_index,speed in enumerate(speed_range):            
            
            X_test = np.array(pred_dict[channel][speed][SNR])
            ood_counter = 0
            for X in X_test:
                #print(X.shape)
                # find the k nearest neighbors:
                knn_prediction = knn.kneighbors(X)
                neighbor_distances = list(np.squeeze(knn_prediction[0]))
                neighbor_indexes = list(np.squeeze(knn_prediction[1]))
                
                ood_decision = []
                class_list_for_this_X = []
                for neighbor_index, neighbor_distance in zip(neighbor_indexes, neighbor_distances):
                    
                    this_neighbor = X_train[neighbor_index]
                    this_neighbor_class = int(y_train[neighbor_index])
                    
                    this_neighbor_distance_from_cluster_center = distance_function(this_neighbor , center_list[this_neighbor_class])
                    this_X_distance_from_cluster_center = distance_function(X , center_list[this_neighbor_class])
                    this_cluster_max_distance = max_distance_list[this_neighbor_class]
                    
                    if this_X_distance_from_cluster_center <= this_neighbor_distance_from_cluster_center and this_X_distance_from_cluster_center < this_cluster_max_distance:
                        ood_decision.append('ID')
                        class_list_for_this_X.append(this_neighbor_class)
                    else:
                        ood_decision.append('OOD')
                
                if 'ID' not in ood_decision:
                    ood_counter += 1
    
    
            # now we have ood_counter for all the samples in the test class, get accuracy
            this_SNR_accuracies.append(ood_counter/X_test.shape[0])
        
        All_SNR_accuracies.append(this_SNR_accuracies)
    
    for i,speed in enumerate(speed_range):
        if speed not in ood_dict:
            ood_dict[speed] = {}
            
        this_list = []
        for j in range(len(SNR_range)):
            this_list.append(round(All_SNR_accuracies[j][i],4))
        
        ood_dict[speed][k_neighbor] = this_list
        # print(this_list)
    # print('-----------------------')

## plotting ood detection rate

In [ ]:
## plotting ood detection rate

x = list(np.arange(21, step=2))
print(x)
y = []

speed_range = [10.0, 50.0, 80.0, 200.0, 400.0]     # test delay range
for speed in speed_range:
    a = ood_dict[speed][5]
    b = ood_dict[speed][10]
    c = ood_dict[speed][15]
    
    y.append([a,b,c])
    print('Delay Spread: ' +str(speed)+ ' ns')
    print(sum(a)/len(a),sum(b)/len(b),sum(c)/len(c))      
  
color_list = ['royalblue','red','ForestGreen','orange','black','violet', 'cyan','gray','yellow','salmon','pink']
legends = ['K=5','K=10','K=15']
markers = ['*','o','+','x']
titles = ['10 ns (ID)','50 ns (ID)','80 ns (ID)','200 ns (OOD)','400 ns (OOD)']
y_limits = [[0,0.2],[0,0.2],[0,0.2],[0,1.1],[0,1.1]]
y_tick_list = [[0,0.1,0.2],[0,0.1,0.2],[0,0.1,0.2],[0,0.5,1],[0,0.5,1]]



plt.rcParams['axes.titley'] = 1    # y is in axes-relative coordinates.
matplotlib.rcParams.update({'font.size': 12})
fig, axes = subplots(figsize=(15,2), nrows=1, ncols=len(y))

print(axes.shape)
print(len(y))
print(len(y[0]))

for i in range(len(y)):
    this_y = y[i]
    ax = axes[i]
    ax.spines[['right', 'top']].set_visible(False)
    for j in range(len(this_y)):
        this_line = this_y[j]
        ax.plot(x, this_line, linewidth=2, color=color_list[j], label=legends[j], marker=markers[j])
    ax.set_ylim(y_limits[i])
    ax.set_xlim([0,21])
    ax.set_xticks([0,10,20])
    ax.set_yticks(y_tick_list[i])
    ax.set_xlabel('Eb/N0 (dB)')
    ax.set_title(titles[i])
    ax.yaxis.grid()
    # put y lable just for fist plot
    if i == 0:
        ax.set_ylabel('OOD Detection \n Rate')
    # put legend just for last subplot
    if i == len(y)-1:
        ax.legend(loc='lower left',bbox_to_anchor=(1.1, 0), ncol=1, borderaxespad=0, edgecolor='black')

fig.tight_layout(pad=1.0)
plt.savefig('foo.pdf', bbox_inches='tight')

# Delay Spread: 10.0 ns
# 0.18454545454545457 0.08653636363636365 0.06461818181818184
# Delay Spread: 50.0 ns
# 0.16411818181818186 0.06005454545454547 0.035590909090909104
# Delay Spread: 80.0 ns
# 0.13891818181818183 0.05184545454545455 0.032136363636363637
# Delay Spread: 200.0 ns
# 0.8806363636363638 0.8271272727272727 0.7926090909090909
# Delay Spread: 400.0 ns
# 0.9658636363636365 0.9474272727272726 0.9357363636363636